<button data-commandLinker-command="progressivis:cleanup_and_run" data-commandlinker-args='{"index": 1}' href="#" class='progressivis-cleanup-and-run-btn'>Run ProgressiVis</button>

In [1]:
from ipyprogressivis.widgets.chaining.constructor import Constructor
from ipyprogressivis.widgets.chaining.utils import create_root, get_header
from ipyprogressivis.widgets.chaining.custom import *
# ***************************************************************************************
# WARNING: This cell must only be executed using the 'Run ProgressiVis' button above.
# Do not execute it in any other way, as the result will not be as expected.
# For the same reason do not copy/paste the contents of this cell to execute it elsewhere
# ***************************************************************************************
header = get_header()
display(header.talker)
display(header.backup)
_ = header.constructor
with header.modules_out:
    display(header.board)
with header.widgets_out:
    display(header.manager)
header.talker.labcommand("notebook:hide-cell-code")
%reload_ext ipyprogressivis.magics
create_root(header.backup)

Talker()

BackupWidget()

## root

In [2]:
# do not run this cell
display(header.constructor)
header.constructor.start_scheduler()
header.talker.labcommand('notebook:hide-cell-code')

Constructor(children=(IntProgress(value=0, description='Starting ProgressiVis ...', max=2, style=ProgressStyle…

Starting scheduler
# Scheduler added module(s): ['sink_1', 'variable_1']


## CSV loader

In [3]:
Constructor.widget('CSV loader', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Made SLA analysis

This notebook analyzes an incident event log dataset with a boolean attribute `made_sla`, which indicates whether an incident met its target SLA. The goal is to examine unique incidents that met and did not meet the SLA, beginning with a visual comparison of both groups. From there, we aim to identify the key factors that influence SLA compliance and determine practical measures that can help increase the number of incidents resolved within the defined SLA targets.

## Snippets

### Sla met/Unmet Vs Unique incident count (on whole data) snippet

In [4]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio

@register_snippet
def progressive_sla_overall_bar(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_sla_overall_bar, "_state"):

        # ----------------------------
        # Progress bar
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )

        unique_label = Label("Unique Incidents: 0")

        uncertainty_label = Label("Uncertainty → SLA Met: ±0 | SLA Unmet: ±0")

        # ----------------------------
        # Bar Chart with error bars
        # ----------------------------
        bar_fig = go.FigureWidget(
            data=[go.Bar(
                x=["SLA Met", "SLA Unmet"],
                y=[0, 0],
                error_y=dict(
                    type='data',
                    array=[0, 0],
                    visible=True
                )
            )]
        )

        bar_fig.update_layout(
            title="Overall SLA Performance (with Uncertainty)",
            xaxis_title="SLA Status",
            yaxis_title="Unique Incident Count",
            width=600,
            height=400
        )

        state = {
            "cursor": 0,
            "batch_size": 50,

            "seen_incidents": {},
            "unique_incidents": set(),

            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "uncertainty_label": uncertainty_label,

            "bar_fig": bar_fig,

            # Track previous counts for uncertainty calculation
            "prev_met_count": 0,
            "prev_unmet_count": 0
        }

        progressive_sla_overall_bar._state = state

        with out:
            display(VBox([
                progress_bar,
                unique_label,
                uncertainty_label,
                bar_fig
            ]))

        # ----------------------------
        # Streaming Loop
        # ----------------------------
        async def process_rows():
            while True:

                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    # Process batch
                    for i in range(start, end):

                        row = table.row(i)

                        incident_id = row.get("number")

                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        sla_raw = row.get("made_sla", False)

                        # YOUR exact logic preserved
                        met_sla = not sla_raw in [True, "TRUE", "True", 1]

                        state["seen_incidents"][incident_id] = met_sla

                    state["cursor"] = end

                    # ----------------------------
                    # Progress update
                    # ----------------------------
                    progress_pct = int(
                        (state["cursor"] / table.nrow) * 100
                    )

                    state["progress_bar"].value = progress_pct

                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # ----------------------------
                    # Compute counts
                    # ----------------------------
                    met_count = 0
                    unmet_count = 0

                    for status in state["seen_incidents"].values():

                        if status:
                            met_count += 1
                        else:
                            unmet_count += 1

                    # ----------------------------
                    # Compute uncertainty
                    # based on batch change
                    # ----------------------------
                    uncertainty_met = abs(
                        met_count - state["prev_met_count"]
                    )

                    uncertainty_unmet = abs(
                        unmet_count - state["prev_unmet_count"]
                    )

                    # Save for next batch
                    state["prev_met_count"] = met_count
                    state["prev_unmet_count"] = unmet_count

                    # Update label
                    state["uncertainty_label"].value = (
                        f"Uncertainty → SLA Met: ±{uncertainty_met} | "
                        f"SLA Unmet: ±{uncertainty_unmet}"
                    )

                    # ----------------------------
                    # Update bar chart with error bars
                    # ----------------------------
                    state["bar_fig"].data[0].y = [
                        met_count,
                        unmet_count
                    ]

                    state["bar_fig"].data[0].error_y.array = [
                        uncertainty_met,
                        uncertainty_unmet
                    ]

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Heatmap diagram for  made_sla Vs priority

In [6]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re

@register_snippet
def progressive_priority_sla_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_priority_sla_dashboard, "_state"):

        # =========================================================
        # TOP: Progress bar
        # =========================================================

        progress_bar = IntProgress(
            value=0,
            min=0,
            max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width': '70%'}
        )

        unique_label = Label("Unique Incidents: 0")

        top_bar = HBox([progress_bar, unique_label])

        # =========================================================
        # HEATMAP
        # =========================================================

        heatmap_fig = go.FigureWidget()

        heatmap_fig.add_trace(
            go.Heatmap(
                z=[[0,0]],
                x=["Breached SLA", "Met SLA"],
                y=[1],
                zmin=0,
                zmax=1,
                colorscale=[
                    [0.0, "#d62728"],
                    [0.5, "#ffaaaa"],
                    [1.0, "#2ca02c"]
                ],
                showscale=True,
                colorbar=dict(title="Breach Fraction"),
                xgap=4,
                ygap=4,
                hovertemplate="%{customdata}<extra></extra>"
            )
        )

        heatmap_fig.update_layout(
            title="SLA Performance by Priority",
            xaxis_title="SLA Status",
            yaxis_title="Priority Level",
            height=400
        )

        # =========================================================
        # HISTOGRAM
        # =========================================================

        histogram_fig = go.FigureWidget()

        histogram_fig.add_trace(
            go.Bar(
                x=[],
                y=[],
                hovertemplate="Priority: %{x}<br>Unique Tickets: %{y}<extra></extra>"
            )
        )

        histogram_fig.update_layout(
            title="Total Unique Tickets by Priority",
            xaxis_title="Priority Level",
            yaxis_title="Unique Ticket Count",
            height=300
        )

        # =========================================================
        # STATE
        # =========================================================

        state = {

            "cursor": 0,
            "batch_size": 50,

            "seen_incidents": {},
            "unique_incidents": set(),

            "prev_breach_counts": defaultdict(int),
            "prev_met_counts": defaultdict(int),

            "fig_heatmap": heatmap_fig,
            "fig_hist": histogram_fig,

            "progress_bar": progress_bar,
            "unique_label": unique_label,
        }

        progressive_priority_sla_dashboard._state = state

        with out:
            display(VBox([
                top_bar,
                heatmap_fig,
                histogram_fig
            ]))

        # =========================================================
        # STREAM LOOP
        # =========================================================

        async def process_rows():

            while True:

                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):

                        row = table.row(i)

                        incident_id = row.get("number")

                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        pr_raw = row.get("priority", 5)

                        if isinstance(pr_raw, str):
                            m = re.search(r"\d+", pr_raw)
                            pr_num = int(m.group(0)) if m else 5
                        else:
                            pr_num = pr_raw

                        sla_raw = row.get("made_sla", False)

                        sla_num = 0 if sla_raw in [True,"TRUE","True",1] else 1

                        state["seen_incidents"][incident_id] = {

                            "priority": pr_num,
                            "made_sla": sla_num
                        }

                    state["cursor"] = end

                    # =====================================
                    # Update progress
                    # =====================================

                    progress_pct = int((state["cursor"]/table.nrow)*100)

                    state["progress_bar"].value = progress_pct

                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # =====================================
                    # Aggregate counts
                    # =====================================

                    priority_totals = defaultdict(int)
                    breach_counts = defaultdict(int)
                    met_counts = defaultdict(int)

                    for inc in state["seen_incidents"].values():

                        pr = inc["priority"]
                        sla = inc["made_sla"]

                        priority_totals[pr] += 1

                        if sla == 0:
                            breach_counts[pr] += 1
                        else:
                            met_counts[pr] += 1

                    priorities = sorted(priority_totals.keys())

                    # =====================================
                    # Heatmap with uncertainty
                    # =====================================

                    z_vals = []
                    custom_data = []

                    for pr in priorities:

                        total = priority_totals[pr]

                        failed = breach_counts[pr]
                        met = met_counts[pr]

                        fraction = failed/total if total else 0

                        # -------------------------
                        # Compute uncertainty
                        # -------------------------

                        prev_failed = state["prev_breach_counts"][pr]
                        prev_met = state["prev_met_counts"][pr]

                        uncert_failed = abs(failed - prev_failed)
                        uncert_met = abs(met - prev_met)

                        stability_failed = (
                            1 - uncert_failed/state["batch_size"]
                        )

                        stability_met = (
                            1 - uncert_met/state["batch_size"]
                        )

                        # save current as previous
                        state["prev_breach_counts"][pr] = failed
                        state["prev_met_counts"][pr] = met

                        z_vals.append([fraction,1-fraction])

                        custom_data.append([

                            f"Priority: {pr}<br>"
                            f"SLA Status: Breached<br>"
                            f"Failed: {failed}<br>"
                            f"Total: {total}<br>"
                            f"Fraction: {fraction:.4f}<br>"
                            f"Uncertainty: ±{uncert_failed} incidents<br>"
                            f"Stability: {stability_failed:.2f}",

                            f"Priority: {pr}<br>"
                            f"SLA Status: Met<br>"
                            f"Met: {met}<br>"
                            f"Total: {total}<br>"
                            f"Fraction: {(1-fraction):.4f}<br>"
                            f"Uncertainty: ±{uncert_met} incidents<br>"
                            f"Stability: {stability_met:.2f}"
                        ])

                    if priorities:

                        state["fig_heatmap"].data[0].z = z_vals
                        state["fig_heatmap"].data[0].y = priorities
                        state["fig_heatmap"].data[0].customdata = custom_data

                    # =====================================
                    # Histogram update
                    # =====================================

                    unique_priority_counts = defaultdict(int)

                    for inc in state["seen_incidents"].values():
                        unique_priority_counts[inc["priority"]] += 1

                    hist_priorities = sorted(unique_priority_counts.keys())
                    hist_counts = [
                        unique_priority_counts[p] for p in hist_priorities
                    ]

                    state["fig_hist"].data[0].x = hist_priorities
                    state["fig_hist"].data[0].y = hist_counts

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Heatmap diagram for made_sla vs impact

In [8]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re

@register_snippet
def progressive_impact_sla_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_impact_sla_dashboard, "_state"):

        # =========================================================
        # Progress UI
        # =========================================================

        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )

        unique_label = Label("Unique Incidents: 0")

        top_bar = HBox([progress_bar, unique_label])

        # =========================================================
        # Heatmap
        # =========================================================

        heatmap_fig = go.FigureWidget()

        heatmap_fig.add_trace(
            go.Heatmap(
                z=[[0,0]],
                x=["Breached SLA","Met SLA"],
                y=["3"],
                zmin=0,
                zmax=1,
                colorscale=[
                    [0.0,"#d62728"],
                    [0.5,"#ffaaaa"],
                    [1.0,"#2ca02c"]
                ],
                showscale=True,
                colorbar=dict(title="Breach Fraction"),
                xgap=4,
                ygap=4,
                hovertemplate="%{customdata}<extra></extra>"
            )
        )

        heatmap_fig.update_layout(
            title="SLA Performance by Impact",
            xaxis_title="SLA Status",
            yaxis_title="Impact Level",
            height=400
        )

        # =========================================================
        # Histogram
        # =========================================================

        histogram_fig = go.FigureWidget()

        histogram_fig.add_trace(
            go.Bar(
                x=[],
                y=[],
                hovertemplate="Impact: %{x}<br>Unique Tickets: %{y}<extra></extra>"
            )
        )

        histogram_fig.update_layout(
            title="Unique Tickets by Impact",
            xaxis_title="Impact Level",
            yaxis_title="Unique Ticket Count",
            height=300
        )

        # =========================================================
        # State
        # =========================================================

        state = {

            "cursor":0,
            "batch_size":50,

            "seen_incidents":{},
            "unique_incidents":set(),

            # uncertainty tracking
            "prev_breach_counts":defaultdict(int),
            "prev_met_counts":defaultdict(int),

            "fig_heatmap":heatmap_fig,
            "fig_hist":histogram_fig,

            "progress_bar":progress_bar,
            "unique_label":unique_label
        }

        progressive_impact_sla_dashboard._state = state

        with out:
            display(VBox([top_bar,heatmap_fig,histogram_fig]))

        # =========================================================
        # Streaming loop
        # =========================================================

        async def process_rows():

            while True:

                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    # -------------------------
                    # Read batch
                    # -------------------------

                    for i in range(start,end):

                        row = table.row(i)

                        incident_id = row.get("number")

                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        imp_raw = row.get("impact",None)

                        if imp_raw is None:
                            imp_num = 3
                        elif isinstance(imp_raw,str):
                            m = re.search(r"\d+",imp_raw)
                            imp_num = int(m.group(0)) if m else 3
                        else:
                            imp_num = imp_raw

                        sla_raw = row.get("made_sla",False)

                        sla_num = 0 if sla_raw in [True,"TRUE","True",1] else 1

                        state["seen_incidents"][incident_id] = {

                            "impact":str(imp_num),
                            "made_sla":sla_num
                        }

                    state["cursor"] = end

                    # -------------------------
                    # Progress update
                    # -------------------------

                    progress_pct = int((state["cursor"]/table.nrow)*100)

                    state["progress_bar"].value = progress_pct

                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # -------------------------
                    # Aggregate
                    # -------------------------

                    impact_totals = defaultdict(int)
                    breach_counts = defaultdict(int)
                    met_counts = defaultdict(int)

                    for inc in state["seen_incidents"].values():

                        imp = inc["impact"]
                        sla = inc["made_sla"]

                        impact_totals[imp]+=1

                        if sla==0:
                            breach_counts[imp]+=1
                        else:
                            met_counts[imp]+=1

                    impact_levels = sorted(impact_totals.keys(),key=int)

                    z_vals=[]
                    custom_data=[]

                    for lvl in impact_levels:

                        total = impact_totals[lvl]

                        failed = breach_counts[lvl]
                        met = met_counts[lvl]

                        fraction = failed/total if total else 0

                        # -------------------------
                        # Uncertainty calculation
                        # -------------------------

                        prev_failed = state["prev_breach_counts"][lvl]
                        prev_met = state["prev_met_counts"][lvl]

                        uncert_failed = abs(failed - prev_failed)
                        uncert_met = abs(met - prev_met)

                        stability_failed = 1 - uncert_failed/state["batch_size"]
                        stability_met = 1 - uncert_met/state["batch_size"]

                        state["prev_breach_counts"][lvl] = failed
                        state["prev_met_counts"][lvl] = met

                        z_vals.append([fraction,1-fraction])

                        custom_data.append([

                            f"Impact Level: {lvl}<br>"
                            f"SLA Status: Breached<br>"
                            f"Failed: {failed}<br>"
                            f"Total: {total}<br>"
                            f"Fraction: {fraction:.4f}<br>"
                            f"Uncertainty: ±{uncert_failed} incidents<br>"
                            f"Stability: {stability_failed:.2f}",

                            f"Impact Level: {lvl}<br>"
                            f"SLA Status: Met<br>"
                            f"Met: {met}<br>"
                            f"Total: {total}<br>"
                            f"Fraction: {(1-fraction):.4f}<br>"
                            f"Uncertainty: ±{uncert_met} incidents<br>"
                            f"Stability: {stability_met:.2f}"
                        ])

                    # -------------------------
                    # Update heatmap
                    # -------------------------

                    if impact_levels:

                        state["fig_heatmap"].data[0].y = impact_levels
                        state["fig_heatmap"].data[0].z = z_vals
                        state["fig_heatmap"].data[0].customdata = custom_data

                    # -------------------------
                    # Update histogram
                    # -------------------------

                    unique_counts = defaultdict(int)

                    for inc in state["seen_incidents"].values():
                        unique_counts[inc["impact"]] += 1

                    hist_levels = sorted(unique_counts.keys(),key=int)
                    hist_counts = [unique_counts[x] for x in hist_levels]

                    state["fig_hist"].data[0].x = hist_levels
                    state["fig_hist"].data[0].y = hist_counts

                await aio.sleep(0.0001)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Heatmap diagram of made_sla Vs Urgency

In [10]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re

@register_snippet
def progressive_urgency_sla_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_urgency_sla_dashboard, "_state"):

        # =========================================================
        # TOP: Progress Bar + Unique Count
        # =========================================================
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )

        unique_label = Label("Unique Incidents: 0")

        top_bar = HBox([progress_bar, unique_label])

        # =========================================================
        # HEATMAP
        # =========================================================
        heatmap_fig = go.FigureWidget()

        heatmap_fig.add_trace(
            go.Heatmap(
                z=[[0,0]],
                x=["Breached SLA", "Met SLA"],
                y=["3"],
                zmin=0, zmax=1,
                colorscale=[
                    [0.0,"#d62728"],
                    [0.5,"#ffaaaa"],
                    [1.0,"#2ca02c"]
                ],
                showscale=True,
                colorbar=dict(title="Breach Fraction"),
                xgap=4,
                ygap=4,
                hovertemplate="%{customdata}<extra></extra>"
            )
        )

        heatmap_fig.update_layout(
            title="SLA Performance by Urgency (Progressive Uncertainty)",
            xaxis_title="SLA Status",
            yaxis_title="Urgency Level",
            height=400
        )

        # =========================================================
        # HISTOGRAM
        # =========================================================
        histogram_fig = go.FigureWidget()

        histogram_fig.add_trace(
            go.Bar(
                x=[],
                y=[],
                hovertemplate=
                "Urgency Level: %{x}<br>"
                "Unique Tickets: %{y}<extra></extra>"
            )
        )

        histogram_fig.update_layout(
            title="Unique Tickets by Urgency",
            xaxis_title="Urgency Level",
            yaxis_title="Unique Ticket Count",
            height=300
        )

        # =========================================================
        # STATE
        # =========================================================
        state = {

            "cursor":0,
            "batch_size":50,

            "seen_incidents":{},
            "unique_incidents":set(),

            # NEW: progressive uncertainty storage
            "previous_fractions":{},

            "fig_heatmap":heatmap_fig,
            "fig_hist":histogram_fig,
            "progress_bar":progress_bar,
            "unique_label":unique_label
        }

        progressive_urgency_sla_dashboard._state = state

        with out:
            display(
                VBox([
                    top_bar,
                    heatmap_fig,
                    histogram_fig
                ])
            )

        # =========================================================
        # STREAM LOOP
        # =========================================================
        async def process_rows():

            while True:

                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    # ===============================
                    # PROCESS BATCH
                    # ===============================
                    for i in range(start,end):

                        row = table.row(i)

                        incident_id = row.get("number")

                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        # urgency extraction
                        urg_raw = row.get("urgency", None)

                        if urg_raw is None:
                            urg_num = 3

                        elif isinstance(urg_raw,str):
                            m = re.search(r"\d+", urg_raw)
                            urg_num = int(m.group(0)) if m else 3

                        else:
                            urg_num = urg_raw

                        # SLA mapping
                        sla_raw = row.get("made_sla", False)

                        sla_num = (
                            0 if sla_raw in [True,"TRUE","True",1]
                            else 1
                        )

                        state["seen_incidents"][incident_id] = {

                            "urgency":str(urg_num),
                            "made_sla":sla_num
                        }

                    state["cursor"] = end

                    # ===============================
                    # UPDATE PROGRESS
                    # ===============================
                    progress_pct = int(
                        (state["cursor"]/table.nrow)*100
                    )

                    state["progress_bar"].value = progress_pct

                    state["unique_label"].value = (
                        f"Unique Incidents: "
                        f"{len(state['unique_incidents'])}"
                    )

                    # ===============================
                    # AGGREGATE HEATMAP
                    # ===============================
                    urgency_totals = defaultdict(int)
                    breach_counts = defaultdict(int)

                    for inc in state["seen_incidents"].values():

                        urg = inc["urgency"]
                        sla = inc["made_sla"]

                        urgency_totals[urg] += 1

                        if sla == 0:
                            breach_counts[urg] += 1


                    urgency_levels = sorted(
                        urgency_totals.keys(),
                        key=int
                    )

                    z_vals = []
                    custom_data = []

                    for lvl in urgency_levels:

                        total = urgency_totals[lvl]
                        failed = breach_counts[lvl]

                        fraction = (
                            failed/total if total>0 else 0
                        )

                        # ===============================
                        # PROGRESSIVE UNCERTAINTY
                        # ===============================
                        prev_fraction = (
                            state["previous_fractions"]
                            .get(lvl, fraction)
                        )

                        uncertainty = abs(
                            fraction - prev_fraction
                        )

                        state["previous_fractions"][lvl] = fraction

                        # ===============================
                        z_vals.append([
                            fraction,
                            1 - fraction
                        ])

                        custom_data.append([

                            f"Urgency Level: {lvl}<br>"
                            f"SLA Status: Breached<br>"
                            f"Breach Fraction: {fraction:.5f}<br>"
                            f"Uncertainty: ±{uncertainty:.5f}<br>"
                            f"Failed: {failed}<br>"
                            f"Total: {total}",

                            f"Urgency Level: {lvl}<br>"
                            f"SLA Status: Met<br>"
                            f"Met Fraction: {(1-fraction):.5f}<br>"
                            f"Uncertainty: ±{uncertainty:.5f}<br>"
                            f"Met: {total-failed}<br>"
                            f"Total: {total}"

                        ])


                    if urgency_levels:

                        state["fig_heatmap"].data[0].y = urgency_levels

                        state["fig_heatmap"].data[0].z = z_vals

                        state["fig_heatmap"].data[0].customdata = custom_data

                        state["fig_heatmap"].data[0].x = [
                            "Breached SLA",
                            "Met SLA"
                        ]


                    # ===============================
                    # UPDATE HISTOGRAM
                    # ===============================
                    unique_counts = defaultdict(int)

                    for incident_id in state["unique_incidents"]:

                        inc = state["seen_incidents"].get(
                            incident_id
                        )

                        if inc:
                            unique_counts[inc["urgency"]] += 1


                    hist_levels = sorted(
                        unique_counts.keys(),
                        key=int
                    )

                    hist_counts = [
                        unique_counts[lvl]
                        for lvl in hist_levels
                    ]

                    state["fig_hist"].data[0].x = hist_levels
                    state["fig_hist"].data[0].y = hist_counts


                await aio.sleep(0.1)


        aio.create_task(process_rows())


    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Group capability visualization using bar chart

In [12]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re
import math

@register_snippet
def progressive_group_sla_capability(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_group_sla_capability, "_state"):

        # ----------------------------
        # Progress UI
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Vertical Bar Chart with Error Bars
        # ----------------------------
        fig = go.FigureWidget()
        fig.add_trace(go.Bar(
            x=[],
            y=[],
            orientation='v',
            hovertemplate="%{customdata}<extra></extra>",
            marker_color='dodgerblue',
            width=0.8,
            error_y=dict(
                type='data',
                array=[],
                visible=True,
                symmetric=False
            )
        ))

        fig.update_layout(
            title="Group Capability Based on SLA (Descending) with Uncertainty",
            xaxis_title="Assignment Group",
            yaxis_title="Capability Score",
            width=1400,
            height=600,
            margin=dict(l=100, r=40, t=80, b=160)
        )

        # ----------------------------
        # State
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},
            "unique_incidents": set(),
            "fig": fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "beta": 0.35,
            "prev_capabilities": defaultdict(float)
        }

        progressive_group_sla_capability._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        # Extract numeric safely
                        def extract_number(val, default=3):
                            if isinstance(val, str):
                                m = re.search(r'\d+', val)
                                return int(m.group(0)) if m else default
                            return val if val is not None else default

                        pr = extract_number(row.get("priority", 5), 5)
                        ur = extract_number(row.get("urgency", 3), 3)
                        im = extract_number(row.get("impact", 3), 3)

                        # SLA logic
                        sla_raw = row.get("made_sla", False)
                        met_sla = not sla_raw in [True, "TRUE", "True", 1]

                        group = row.get("assignment_group", "Unknown")

                        state["seen_incidents"][incident_id] = {
                            "group": group,
                            "met_sla": met_sla,
                            "priority": pr,
                            "urgency": ur,
                            "impact": im
                        }

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # ----------------------------
                    # Aggregate capability per group
                    # ----------------------------
                    group_data = defaultdict(
                        lambda: {"capability":0, "met":0, "breach":0, "total":0}
                    )

                    for inc_id in state["unique_incidents"]:
                        inc = state["seen_incidents"].get(inc_id)
                        if not inc:
                            continue

                        g = inc["group"]
                        pr = inc["priority"]
                        ur = inc["urgency"]
                        im = inc["impact"]

                        weight = 1 / math.log(pr + ur + im + 1e-6)
                        weight = min(weight, 10)

                        group_data[g]["total"] += 1

                        if inc["met_sla"]:
                            group_data[g]["capability"] += weight
                            group_data[g]["met"] += 1
                        else:
                            group_data[g]["capability"] -= state["beta"] * weight
                            group_data[g]["breach"] += 1

                    # ----------------------------
                    # Filter and sort
                    # ----------------------------
                    filtered_groups = {
                        g: d for g, d in group_data.items()
                        if d["capability"] >= 40 or d["capability"] <= -40
                    }

                    sorted_groups = sorted(
                        filtered_groups.items(),
                        key=lambda item: item[1]["capability"],
                        reverse=True
                    )

                    x_vals = []
                    y_vals = []
                    custom_data = []
                    err_y = []

                    for g, d in sorted_groups:
                        x_vals.append(g)
                        y_vals.append(d["capability"])

                        # Calculate uncertainty (change from previous batch)
                        prev = state["prev_capabilities"].get(g, 0)
                        uncertainty = abs(d["capability"] - prev)
                        state["prev_capabilities"][g] = d["capability"]

                        # For negative bars, extend error in opposite direction
                        if d["capability"] >= 0:
                            err_y.append([0, uncertainty])  # from bar top
                        else:
                            err_y.append([uncertainty, 0])  # from bar top downward

                        total = d["total"]
                        met = d["met"]
                        breach = d["breach"]
                        frac_met = met / total if total > 0 else 0
                        frac_breach = breach / total if total > 0 else 0

                        hovertext = (
                            f"Group: {g}<br>"
                            f"Capability: {d['capability']:.2f}<br>"
                            f"Total Tickets: {total}<br>"
                            f"Met SLA: {met} ({frac_met:.4f})<br>"
                            f"Breach SLA: {breach} ({frac_breach:.4f})<br>"
                            f"Uncertainty: {uncertainty:.2f}"
                        )

                        custom_data.append(hovertext)

                    # ----------------------------
                    # Update figure
                    # ----------------------------
                    if x_vals:
                        state["fig"].data[0].x = x_vals
                        state["fig"].data[0].y = y_vals
                        state["fig"].data[0].customdata = custom_data
                        state["fig"].data[0].error_y = dict(
                            type='data',
                            array=[u[1] for u in err_y],
                            arrayminus=[u[0] for u in err_y],
                            visible=True,
                            symmetric=False
                        )

                        state["fig"].update_layout(
                            xaxis={'tickangle':-45, 'automargin':True},
                            yaxis={'title':'Capability Score'}
                        )

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### SLA met and Unmet's top 5 categories

In [16]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict

@register_snippet
def progressive_sla_category_bars(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_sla_category_bars, "_state"):

        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # MET BAR CHART
        # ----------------------------
        met_fig = go.FigureWidget(
            data=[go.Bar(
                x=[],
                y=[],
                error_x=dict(type='data', array=[], visible=True),
                customdata=[],
                hovertemplate=
                "Category: %{y}<br>"
                "Met Count: %{x}<br>"
                "Average Priority: %{customdata:.2f}<br>"
                "±Uncertainty: %{error_x.array:.2f}<extra></extra>",
                orientation='h'
            )]
        )
        met_fig.update_layout(
            title="Top 5 SLA Met Categories",
            width=500,
            height=400,
            yaxis=dict(autorange="reversed")
        )

        # ----------------------------
        # UNMET BAR CHART
        # ----------------------------
        unmet_fig = go.FigureWidget(
            data=[go.Bar(
                x=[],
                y=[],
                error_x=dict(type='data', array=[], visible=True),
                customdata=[],
                hovertemplate=
                "Category: %{y}<br>"
                "Unmet Count: %{x}<br>"
                "Average Priority: %{customdata:.2f}<br>"
                "±Uncertainty: %{error_x.array:.2f}<extra></extra>",
                orientation='h'
            )]
        )
        unmet_fig.update_layout(
            title="Top 5 SLA Unmet Categories",
            width=500,
            height=400,
            yaxis=dict(autorange="reversed")
        )

        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},
            "unique_incidents": set(),
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "met_fig": met_fig,
            "unmet_fig": unmet_fig
        }

        progressive_sla_category_bars._state = state

        with out:
            display(VBox([
                top_bar,
                HBox([met_fig, unmet_fig])
            ]))

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        category = row.get("category", "Unknown")
                        sla_raw = row.get("made_sla", False)
                        met_sla = not sla_raw in [True, "TRUE", "True", 1]

                        priority_raw = row.get("priority", 0)
                        try:
                            if isinstance(priority_raw, str):
                                priority = float(priority_raw.split("-")[0].strip())
                            else:
                                priority = float(priority_raw)
                        except:
                            priority = 0

                        state["seen_incidents"][incident_id] = {
                            "category": category,
                            "met_sla": met_sla,
                            "priority": priority
                        }

                    state["cursor"] = end

                    # Update progress
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # Aggregate
                    met_counts = defaultdict(int)
                    unmet_counts = defaultdict(int)
                    met_priority_sum = defaultdict(float)
                    unmet_priority_sum = defaultdict(float)
                    total_incidents = 0

                    for inc in state["seen_incidents"].values():
                        cat = inc["category"]
                        pr = inc["priority"]
                        total_incidents += 1

                        if inc["met_sla"]:
                            met_counts[cat] += 1
                            met_priority_sum[cat] += pr
                        else:
                            unmet_counts[cat] += 1
                            unmet_priority_sum[cat] += pr

                    # ----------------------------
                    # Sort + Take Top 5 (MET)
                    # ----------------------------
                    met_sorted = sorted(
                        met_counts.items(),
                        key=lambda x: x[1],
                        reverse=True
                    )[:5]

                    met_labels = [x[0] for x in met_sorted]
                    met_values = [x[1] for x in met_sorted]
                    met_avg = [
                        met_priority_sum[cat] / met_counts[cat]
                        for cat in met_labels
                    ]
                    # Calculate uncertainty (SE)
                    met_se = [
                        (v*(1 - v/total_incidents))**0.5 if total_incidents > 0 else 0
                        for v in met_values
                    ]

                    state["met_fig"].data[0].x = met_values
                    state["met_fig"].data[0].y = met_labels
                    state["met_fig"].data[0].customdata = met_avg
                    state["met_fig"].data[0].error_x.array = met_se

                    # ----------------------------
                    # Sort + Take Top 5 (UNMET)
                    # ----------------------------
                    unmet_sorted = sorted(
                        unmet_counts.items(),
                        key=lambda x: x[1],
                        reverse=True
                    )[:5]

                    unmet_labels = [x[0] for x in unmet_sorted]
                    unmet_values = [x[1] for x in unmet_sorted]
                    unmet_avg = [
                        unmet_priority_sum[cat] / unmet_counts[cat]
                        for cat in unmet_labels
                    ]
                    # Calculate uncertainty (SE)
                    unmet_se = [
                        (v*(1 - v/total_incidents))**0.5 if total_incidents > 0 else 0
                        for v in unmet_values
                    ]

                    state["unmet_fig"].data[0].x = unmet_values
                    state["unmet_fig"].data[0].y = unmet_labels
                    state["unmet_fig"].data[0].customdata = unmet_avg
                    state["unmet_fig"].data[0].error_x.array = unmet_se

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Priority Distribution among top 4 Categories

In [18]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re
import math

@register_snippet
def progressive_category_priority_bar_filtered(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    allowed_categories = {"Category 42", "Category 53", "Category 26", "Category 46"}

    if not hasattr(progressive_category_priority_bar_filtered, "_state"):

        # ----------------------------
        # Progress bar
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Grouped bar chart with error bars
        # ----------------------------
        fig = go.FigureWidget()
        colors = {1: 'red', 2: 'orange', 3: 'green'}

        for p in [1, 2, 3]:
            fig.add_trace(go.Bar(
                x=[],
                y=[],
                name=f'Priority {p}',
                marker_color=colors[p],
                error_y=dict(type='data', array=[], visible=True)
            ))

        fig.update_layout(
            title="Unique Incidents by Selected Categories and Priority (with 95% CI)",
            xaxis_title="Category",
            yaxis_title="Unique Incident Count",
            barmode='group',
            width=900,
            height=500,
            margin=dict(l=80, r=40, t=80, b=120)
        )

        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},
            "unique_incidents": set(),
            "fig": fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label
        }

        progressive_category_priority_bar_filtered._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        category = str(row.get("category", "Unknown"))
                        if category not in allowed_categories:
                            continue

                        pr_raw = row.get("priority", 3)
                        try:
                            if isinstance(pr_raw, str):
                                m = re.match(r'(\d+)', pr_raw)
                                priority = int(m.group(1)) if m else 3
                            else:
                                priority = int(pr_raw)
                        except:
                            priority = 3

                        state["seen_incidents"][incident_id] = {
                            "category": category,
                            "priority": priority
                        }

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # ----------------------------
                    # Aggregate counts
                    # ----------------------------
                    cat_pr_count = defaultdict(lambda: defaultdict(int))
                    cat_total = defaultdict(int)

                    for inc in state["seen_incidents"].values():
                        cat = inc["category"]
                        pr = inc["priority"]
                        cat_pr_count[cat][pr] += 1
                        cat_total[cat] += 1

                    # ----------------------------
                    # Sort categories by total DESC
                    # ----------------------------
                    sorted_categories = sorted(
                        cat_total.keys(),
                        key=lambda c: cat_total[c],
                        reverse=True
                    )

                    # ----------------------------
                    # Update figure with error bars (95% CI)
                    # ----------------------------
                    for idx, p in enumerate([1, 2, 3]):
                        y_vals = []
                        error_vals = []

                        for cat in sorted_categories:
                            count = cat_pr_count[cat].get(p, 0)
                            total = cat_total[cat]

                            y_vals.append(count)

                            # Compute 95% confidence interval using binomial SE
                            if total > 0:
                                prop = count / total
                                se = math.sqrt(prop * (1 - prop) / total)
                                error = 1.96 * se * total  # convert back to count scale
                            else:
                                error = 0
                            error_vals.append(error)

                        hover_text = [
                            f"Category: {cat}<br>"
                            f"Priority: {p}<br>"
                            f"Unique Incidents: {y}<br>"
                            f"95% CI ±{err:.2f}"
                            for cat, y, err in zip(sorted_categories, y_vals, error_vals)
                        ]

                        state["fig"].data[idx].x = sorted_categories
                        state["fig"].data[idx].y = y_vals
                        state["fig"].data[idx].error_y.array = error_vals
                        state["fig"].data[idx].customdata = hover_text
                        state["fig"].data[idx].hovertemplate = "%{customdata}<extra></extra>"

                await aio.sleep(0.0001)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Knowledge Used distribution among major categories

In [21]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import math

@register_snippet
def progressive_category_knowledge_bar(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    allowed_categories = {"Category 42", "Category 53", "Category 26", "Category 46"}

    if not hasattr(progressive_category_knowledge_bar, "_state"):

        # ----------------------------
        # Progress UI
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Bar chart
        # ----------------------------
        fig = go.FigureWidget()
        colors = {'used':'green', 'not_used':'red'}

        for k in ['used','not_used']:
            fig.add_trace(go.Bar(
                x=[],
                y=[],
                name=f'Knowledge {k}',
                marker_color=colors[k],
                error_y=dict(type='data', array=[], visible=True)
            ))

        fig.update_layout(
            title="Unique Incidents by Selected Categories and Knowledge Usage with Uncertainty",
            xaxis_title="Category",
            yaxis_title="Unique Incident Count",
            barmode='group',
            width=900,
            height=500,
            margin=dict(l=80, r=40, t=80, b=120)
        )

        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},
            "unique_incidents": set(),
            "fig": fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label
        }

        progressive_category_knowledge_bar._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        state["unique_incidents"].add(incident_id)

                        category = str(row.get("category", "Unknown"))
                        if category not in allowed_categories:
                            continue

                        knowledge_used = row.get("knowledge", False)
                        knowledge_status = 'used' if knowledge_used else 'not_used'

                        state["seen_incidents"][incident_id] = {
                            "category": category,
                            "knowledge_status": knowledge_status
                        }

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"]/table.nrow)*100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # ----------------------------
                    # Aggregate counts
                    # ----------------------------
                    cat_k_count = defaultdict(lambda: defaultdict(int))
                    cat_total = defaultdict(int)

                    for inc in state["seen_incidents"].values():
                        cat = inc["category"]
                        k = inc["knowledge_status"]

                        cat_k_count[cat][k] += 1
                        cat_total[cat] += 1  # total unique incidents per category

                    # ----------------------------
                    # SORT categories by total DESC
                    # ----------------------------
                    sorted_categories = sorted(
                        cat_total.keys(),
                        key=lambda c: cat_total[c],
                        reverse=True
                    )

                    # ----------------------------
                    # Compute 95% confidence interval using binomial approximation
                    # ----------------------------
                    def compute_ci(successes, total, z=1.96):
                        if total == 0:
                            return 0
                        p = successes / total
                        se = math.sqrt(p * (1 - p) / total)
                        return z * se * total  # scale by total to match bar height

                    # ----------------------------
                    # Update chart with error bars
                    # ----------------------------
                    for idx, k in enumerate(['used','not_used']):

                        y_vals = [
                            cat_k_count[cat].get(k, 0)
                            for cat in sorted_categories
                        ]

                        error_vals = [
                            compute_ci(cat_k_count[cat].get(k, 0), cat_total[cat])
                            for cat in sorted_categories
                        ]

                        hover_text = [
                            f"Category: {cat}<br>"
                            f"Knowledge: {k}<br>"
                            f"Unique Incidents: {y}<br>"
                            f"95% CI: ±{err:.1f}"
                            for cat, y, err in zip(sorted_categories, y_vals, error_vals)
                        ]

                        state["fig"].data[idx].x = sorted_categories
                        state["fig"].data[idx].y = y_vals
                        state["fig"].data[idx].customdata = hover_text
                        state["fig"].data[idx].hovertemplate = "%{customdata}<extra></extra>"
                        state["fig"].data[idx].error_y.array = error_vals
                        state["fig"].data[idx].error_y.visible = True

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Impact of Knowledge Base on Meeting target SLA or Breaching target SLA

In [23]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label, Button
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re

@register_snippet
def progressive_knowledge_sla_odds(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_knowledge_sla_odds, "_state"):

        # ----------------------------
        # Progress bar + label
        # ----------------------------
        progress_bar = IntProgress(value=0, min=0, max=100,
                                   description='Streaming:', bar_style='info', layout={'width':'70%'})
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Stacked bar chart
        # ----------------------------
        fig = go.FigureWidget()
        fig.add_trace(go.Bar(
            x=["KB Used", "KB Not Used"], y=[0,0],
            name="Met SLA", marker_color='green', hovertemplate="%{customdata}<extra></extra>"
        ))
        fig.add_trace(go.Bar(
            x=["KB Used", "KB Not Used"], y=[0,0],
            name="Breached SLA", marker_color='red', hovertemplate="%{customdata}<extra></extra>"
        ))
        fig.update_layout(
            barmode='stack',
            title="Knowledge Usage vs SLA",
            xaxis_title="Knowledge Used",
            yaxis_title="Incident Count",
            width=800, height=500
        )

        # ----------------------------
        # Button for odds calculation
        # ----------------------------
        odds_button = Button(description="Compute Odds & Relative Score", button_style='info')
        odds_out = Output()

        # ----------------------------
        # Persistent state
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},  # incident_id -> latest row
            "unique_incidents": set(),
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "fig": fig,
            "odds_button": odds_button,
            "odds_out": odds_out
        }

        progressive_knowledge_sla_odds._state = state

        # ----------------------------
        # Display widgets
        # ----------------------------
        with out:
            display(VBox([
                top_bar,
                fig,
                odds_button,
                odds_out
            ]))

        # ----------------------------
        # Button click callback
        # ----------------------------
        def compute_odds(b):
            counts = defaultdict(int)
            for inc in state["seen_incidents"].values():
                kb_used = bool(inc.get("knowledge", False))
                met_sla = inc.get("met_sla", False)
                if kb_used and met_sla:
                    counts["KBUsed_Met"] += 1
                elif kb_used and not met_sla:
                    counts["KBUsed_Breach"] += 1
                elif not kb_used and met_sla:
                    counts["KBNotUsed_Met"] += 1
                elif not kb_used and not met_sla:
                    counts["KBNotUsed_Breach"] += 1

            # Odds ratio = (a/b)/(c/d)
            a = counts.get("KBUsed_Met",0)
            b = counts.get("KBUsed_Breach",0)
            c = counts.get("KBNotUsed_Met",0)
            d = counts.get("KBNotUsed_Breach",0)
            odds_ratio = ((a/b) / (c/d)) if b>0 and c>0 and d>0 else float('nan')
            # Relative probability
            prob_met_given_kb = a / (a+b) if (a+b)>0 else float('nan')
            prob_met_given_no_kb = c / (c+d) if (c+d)>0 else float('nan')
            relative_prob = prob_met_given_kb / prob_met_given_no_kb if prob_met_given_no_kb>0 else float('nan')

            with odds_out:
                odds_out.clear_output()
                print(f"Odds Ratio (SLA met if knowledge used): {odds_ratio:.3f}")
                print(f"Relative Probability: {relative_prob:.3f}")
                print(f"Counts: KBUsed Met {a}, Breach {b}; KBNotUsed Met {c}, Breach {d}")

        odds_button.on_click(compute_odds)

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)
                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        # Track unique incident
                        state["unique_incidents"].add(incident_id)

                        # Knowledge field
                        knowledge = row.get("knowledge", False)

                        # SLA: True = breached, False = met
                        sla_raw = row.get("made_sla", False)
                        met_sla = not sla_raw in [True,"TRUE","True",1]

                        # Priority extraction (numeric from "3 - Moderate")
                        pr_raw = row.get("priority",0)
                        try:
                            pr = int(str(pr_raw).split("-")[0].strip())
                        except:
                            pr = 0

                        state["seen_incidents"][incident_id] = {
                            "knowledge": knowledge,
                            "met_sla": met_sla,
                            "priority": pr
                        }

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"]/table.nrow)*100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # ----------------------------
                    # Aggregate for chart
                    # ----------------------------
                    agg = {
                        "KBUsed_Met": 0, "KBUsed_Breach": 0,
                        "KBNotUsed_Met": 0, "KBNotUsed_Breach": 0,
                        "KBUsed_PrioritySum":0, "KBUsed_Count":0,
                        "KBNotUsed_PrioritySum":0, "KBNotUsed_Count":0
                    }

                    for inc in state["seen_incidents"].values():
                        kb_used = bool(inc["knowledge"])
                        met_sla = inc["met_sla"]
                        pr = inc["priority"]
                        if kb_used and met_sla:
                            agg["KBUsed_Met"] +=1
                            agg["KBUsed_PrioritySum"] += pr
                            agg["KBUsed_Count"] +=1
                        elif kb_used and not met_sla:
                            agg["KBUsed_Breach"] +=1
                            agg["KBUsed_PrioritySum"] += pr
                            agg["KBUsed_Count"] +=1
                        elif not kb_used and met_sla:
                            agg["KBNotUsed_Met"] +=1
                            agg["KBNotUsed_PrioritySum"] += pr
                            agg["KBNotUsed_Count"] +=1
                        elif not kb_used and not met_sla:
                            agg["KBNotUsed_Breach"] +=1
                            agg["KBNotUsed_PrioritySum"] += pr
                            agg["KBNotUsed_Count"] +=1

                    # Update hover template with average priority
                    kb_used_met_avg = (agg["KBUsed_PrioritySum"]/agg["KBUsed_Count"]) if agg["KBUsed_Count"]>0 else 0
                    kb_used_breach_avg = (agg["KBUsed_PrioritySum"]/agg["KBUsed_Count"]) if agg["KBUsed_Count"]>0 else 0
                    kb_notused_met_avg = (agg["KBNotUsed_PrioritySum"]/agg["KBNotUsed_Count"]) if agg["KBNotUsed_Count"]>0 else 0
                    kb_notused_breach_avg = (agg["KBNotUsed_PrioritySum"]/agg["KBNotUsed_Count"]) if agg["KBNotUsed_Count"]>0 else 0

                    state["fig"].data[0].y = [agg["KBUsed_Met"], agg["KBNotUsed_Met"]]
                    state["fig"].data[1].y = [agg["KBUsed_Breach"], agg["KBNotUsed_Breach"]]

                    state["fig"].data[0].customdata = [
                        f"Count: {agg['KBUsed_Met']}<br>Avg Priority: {kb_used_met_avg:.2f}",
                        f"Count: {agg['KBNotUsed_Met']}<br>Avg Priority: {kb_notused_met_avg:.2f}"
                    ]
                    state["fig"].data[1].customdata = [
                        f"Count: {agg['KBUsed_Breach']}<br>Avg Priority: {kb_used_breach_avg:.2f}",
                        f"Count: {agg['KBNotUsed_Breach']}<br>Avg Priority: {kb_notused_breach_avg:.2f}"
                    ]

                await aio.sleep(0.1)

        aio.create_task(process_rows())

    return SnippetResult(output_module=input_module, output_slot=input_slot, widget=out)

## Made_SLA VS Unique_Incidents

In [5]:
Constructor.widget('Snippet', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights
- Initially, most incidents met the target SLA, as indicated by made_sla = false (meaning the SLA was not exceeded).
- However, over time, an increasing number of incidents began failing to meet the SLA.
- This trend is clearly visible in the graph above.

**To understand the reasons behind this shift, we will analyze the relationship between made_sla and other key attributes such as priority, urgency, and impact. The objective is to determine how these factors influence SLA compliance.**

## Heatmap of SLA compliance by Priority

In [7]:
Constructor.widget('Snippet', 1)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Heatmap of SLA compliance by Impact

In [9]:
Constructor.widget('Snippet', 2)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Heatmap of SLA compliance by Urgency

In [11]:
Constructor.widget('Snippet', 3)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insight

- Incidents with high priority, high urgency, and high impact (lower numerical values) have a higher rate of SLA compliance.
- This indicates that critical incidents receive immediate attention and are resolved efficiently.
- However, these high-severity incidents represent a relatively small portion of the total workload.
- A larger share of SLA breaches comes from incidents with lower priority, urgency, and impact.

**To better understand whether this is due to workload distribution or group performance limitations, we introduce a group capability score that measures how effectively each group handles its assigned incidents.**

### Capability Score Definition

We define a **severity-based score** for each incident:

$$
S = \frac{10}{\log(priority + impact + urgency)}
$$

- Higher-severity incidents (lower values) receive higher weights.

We then consider two cases:

- **SLA Met (`made_sla = false`)**:

$$
S_{met} = \frac{10}{\log(priority + impact + urgency)}
$$

- **SLA Breached (`made_sla = true`)**:

$$
S_{breach} = \frac{10}{\log(priority + impact + urgency)}
$$

To reduce the penalty effect, we introduce a factor:

$$
\beta = 0.15
$$

---

## Capability Update Rule

For each incident assigned to a group, we update capability:

$$
\text{capability} += 
\begin{cases} 
S_{met}, & \text{if } made\_sla = false \\
-\beta \cdot S_{breach}, & \text{if } made\_sla = true 
\end{cases}
$$

Finally, we ensure:

$$
\text{capability} = \max(0, \text{capability})
$$

- This prevents negative capability scores.

---

## Analysis

- Summing scores across all incidents per group gives a measure of **group performance**.  
- Comparing capability scores and the number of assigned incidents highlights **workload imbalances**, **performance bottlenecks**, and opportunities for **operational improvement**.

## Group capability visualization using Bar chart

In [13]:
Constructor.widget('Snippet', 4)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights

- **Group 70:** Initially performed very well. However, it was assigned a large number of incidents (~9,400), and as workload increased, its capability score started to decrease. This indicates that high-performing groups can see reduced efficiency under heavy load.

- **Group 64:** Performed poorly, with only 0.16% of incidents meeting SLA compliance. This group consistently underperforms compared to others.

- **Overall Performance:** Most groups have around 50% of incidents meeting SLA.

### Key Takeaways

- Group 70 is a strong performer, but its efficiency suffers under high load. Optimizing incident assignment to balance workload can help more incidents meet SLA targets.

- Group 64 demonstrates consistently low performance. Consider re-evaluating or replacing this group.

- **Recommendation:** Implement an **epsilon-greedy assignment strategy**:
  - With probability ε, assign tickets randomly to a group.
  - With probability 1−ε, assign tickets greedily to the best-performing group.  
  This approach balances exploration and exploitation, improving SLA compliance while preventing overload on top-performing groups.

## Category Analysis of SLA Compliance

- Next, we analyze which incident categories contribute to **breaching SLAs** and **meeting SLAs**.  

- In this case, we are only interested in **identifying only the top 5 categories** of SLA breaches and compliances.


## Bar chart depicting the top 5 Categories Breaching SLA and Meeting SLA

In [17]:
Constructor.widget('Snippet', 6)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights
- category 42, 53, 26, 46 forms major share of categories that meets SLA, as well as Breaches SLA.
- Improvement : More dedicated groups can be assigned to these categories
- we check the priorities in each of this category (number of unique incidents in priority 1,2,3) 
- and knowledge base in each of these category. (number of unique incidents for which knowledge base exists), no. of unique incidents for which knowledge base doesn't exists).

## Priority Distribution among major Categories

In [19]:
Constructor.widget('Snippet', 7)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Insights
- priority 3 incidents counts are the most and hence we have most breached sla's in these categories
- To improve we can train dedicated group on these major categories 

## Knowledge Used Distribution among major Categories

In [22]:
Constructor.widget('Snippet', 8)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Insights
- For all of the major categories we see that a document from knowledge base was not used
- This is another reason why target SLA were not met for these categories
- Knowledge was not used possibly because of its non-existence or difficulty to find it. 
### Suggestions 
- once we train dedicated group, we can suggest them to create a RCA of incident and its resolution, so the knowledge base can be used later.

## Impact of knowledge base on meeting SLA 
- But before we make the insight that using knowledge base increases the chance of meeting SLA, we need to make analysis on the knowledge base
- We need to calculate the fraction for which knowledge base was used and SLA was met, and knowledge base was used and SLA was breached
- Similarly we need to calculate the fraction for which knowledge base was not used and SLA was met, and knowledge base was not used and SLA was breached.
- Having this fraction visualized will make our insight more concrete. 

# SLA Analysis: Knowledge Base Impact

This section explains how to quantify the effect of **Knowledge Base (KB) usage** on SLA compliance using a **contingency table**, **Odds Ratio (OR)**, and **Relative Probability / Risk Ratio (RP)**.

---

## Contingency Table (2×2)

We organize incidents into a 2×2 table:

|               | SLA Met | SLA Unmet |
|---------------|---------|-----------|
| **KB Used**    | a       | b         |
| **KB Not Used**| c       | d         |

Where:  

- **a** = Number of incidents where KB was used & SLA met  
- **b** = Number of incidents where KB was used & SLA breached  
- **c** = Number of incidents where KB was not used & SLA met  
- **d** = Number of incidents where KB was not used & SLA breached  

This table is the basis for calculating **Odds Ratio** and **Relative Probability**.

---

## 1. Odds Ratio (OR)

**Formula:**

$$
OR = \frac{a / b}{c / d} = \frac{a \cdot d}{b \cdot c}
$$

**Interpretation:**

- `OR = 1` → KB usage has no effect on meeting SLA  
- `OR > 1` → Using KB **increases the odds** of meeting SLA  
- `OR < 1` → Using KB **decreases the odds** of meeting SLA  

> If KB is very effective, OR can be significantly greater than 1.

---

## 2. Relative Probability / Risk Ratio (RP)

**Formula:**

$$
RP = \frac{P(\text{SLA met | KB used})}{P(\text{SLA met | KB not used})} = \frac{a / (a + b)}{c / (c + d)}
$$

**Interpretation:**

- `RP = 1` → KB usage does **not change** the probability of meeting SLA  
- `RP > 1` → KB usage **increases** the probability of meeting SLA  
- `RP < 1` → KB usage **decreases** the probability of meeting SLA  

> RP can exceed 1 depending on how effectively KB improves SLA resolution.


## Impact of Knowledge base on Meeting target SLA

In [24]:
Constructor.widget('Snippet', 9)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Insights
- we can see that using KB increases the odds by 6.243
- and using KB increases the probability of meeting SLA 2.403 times
- This solidifies our suggestion to train a dedicated group on major categories 42, 53, 26 and 46, and on meeting SLA and resolution, create RCA in knowledge base.

## Summary

### Insights
- Incidents with high priority, urgency, and impact receive more attention and are resolved faster.  
- Most incidents are low-priority, leading to a higher number of SLA breaches.  
- Categories 26, 42, 53, and 46 contribute the most to SLA breaches, mainly due to many low-priority incidents.  
- Group 70 is overloaded, causing a decline in performance over time.  
- Group 64 consistently underperforms and may need to be replaced.  
- Using the Knowledge Base doubles the probability of meeting SLA and increases the odds sixfold, showing a significant positive impact.  

### Suggestions
- Efficiently distribute tasks using an **epsilon-greedy strategy**, balancing workload across groups.  
- Train a dedicated group to handle incidents in categories 26, 42, 53, and 46.  
- This group should perform **Root Cause Analysis (RCA)** for recurring issues and update the Knowledge Base to improve resolution efficiency.  
- Continuously monitor SLA metrics to ensure improvements from RCA and KB updates are effective.  